---
## ⚙️ Step 1: Install & Import

In [ ]:
%pip install openai pydantic --quiet

In [ ]:
import os
import re
import json
import datetime
from typing import Literal, Optional
from openai import AzureOpenAI
from getpass import getpass
from pydantic import BaseModel, Field, field_validator, model_validator, ValidationError
from enum import Enum

print("✅ Imports OK")

---
## 🔑 Step 2: Configure Azure OpenAI Client

In [ ]:
AZURE_OPENAI_ENDPOINT = input("Endpoint (e.g. https://xxx.openai.azure.com/): ").strip()
AZURE_OPENAI_API_KEY  = getpass("API Key: ")
AZURE_DEPLOYMENT_NAME = input("Deployment name (e.g. gpt-4o): ").strip()
AZURE_API_VERSION     = "2024-08-01-preview"

client = AzureOpenAI(
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    api_key        = AZURE_OPENAI_API_KEY,
    api_version    = AZURE_API_VERSION,
)

print("✅ Azure OpenAI client initialised successfully!")

---
## 📦 Step 3: Load Schema and Model (self-contained)

In [ ]:
class TaxDocument(BaseModel):
    entity_name:           Optional[str]   = Field(default=None)
    tax_period:            Optional[str]   = Field(default=None)
    transaction_type:      Optional[str]   = Field(default=None)
    jurisdiction:          Optional[str]   = Field(default=None)
    liability_amount:      Optional[float] = Field(default=None)
    currency: Literal["INR","USD","EUR","GBP","AED","SGD","Unknown"] = Field(default="INR")
    applicable_section:    Optional[str]   = Field(default=None)
    extraction_confidence: float           = Field(ge=0.0, le=1.0)
    missing_fields:        list[str]       = Field(default_factory=list)
    extraction_notes:      Optional[str]   = Field(default=None)
    human_review_required: bool            = Field(default=False)

    @field_validator("currency", mode="before")
    @classmethod
    def normalise_currency(cls, v):
        if v is None: return "INR"
        mapping = {"rupee":"INR","rupees":"INR","rs":"INR","inr":"INR","₹":"INR",
                   "dollar":"USD","dollars":"USD","usd":"USD","$":"USD",
                   "euro":"EUR","eur":"EUR","€":"EUR","pound":"GBP","gbp":"GBP","£":"GBP",
                   "dirham":"AED","aed":"AED","sgd":"SGD"}
        return mapping.get(str(v).lower().strip(), str(v).upper() if str(v).upper() in ["INR","USD","EUR","GBP","AED","SGD"] else "Unknown")

    @field_validator("liability_amount", mode="before")
    @classmethod
    def parse_amount(cls, v):
        if v is None: return None
        if isinstance(v, (int, float)): return float(v)
        s = str(v).lower().replace(",","").replace("₹","").replace("rs","").strip()
        m = re.search(r"([\d.]+)\s*l(?:akh)?", s)
        if m: return float(m.group(1)) * 100000
        m = re.search(r"([\d.]+)\s*cr(?:ore)?", s)
        if m: return float(m.group(1)) * 10000000
        m = re.search(r"[\d.]+", s)
        return float(m.group()) if m else None

    @model_validator(mode="after")
    def auto_set_review_flag(self):
        critical = {"entity_name", "tax_period", "transaction_type"}
        if self.extraction_confidence < 0.75 or critical.intersection(set(self.missing_fields)):
            self.human_review_required = True
        return self


try:
    with open("day4_schema.json") as f:
        EXTRACTION_FUNCTION_SCHEMA = json.load(f)
    print("✅ Schema loaded from day4_schema.json")
except FileNotFoundError:
    print("⚠️  Run Notebook 4A to generate day4_schema.json")

---
## 🏗️ Step 4: Build the Full Pipeline

Three failure modes, three responses:

| Failure | Response |
|---|---|
| Parse failure (JSON error) | Retry once with stronger instruction. Log if still fails. |
| Validation failure (wrong types/values) | Return partial dict with bad fields noted in `missing_fields`. |
| Low confidence (< 0.75) | Pass through — but set `human_review_required = True`. |

Two routing outcomes:
- `extraction_confidence ≥ 0.75` and no critical missing fields → **DB insert queue**
- Otherwise → **Human review queue** with pre-populated form

In [ ]:
EXTRACTION_SYSTEM_PROMPT = (
    "You are a senior tax compliance analyst. Extract structured fields from the document.\n"
    "Never hallucinate values. Null fields you cannot extract. List them in missing_fields."
)

STRICT_EXTRACTION_PROMPT = (
    "You are a tax extraction system. Return ONLY the function call. "
    "No text. No explanation. Only JSON function arguments matching the schema exactly."
)


class RouteOutcome(str, Enum):
    DB_INSERT    = "db_insert"
    HUMAN_REVIEW = "human_review"
    FAILED       = "failed"


def _call_extraction_api(document_text, system_prompt):
    """Single API call — returns raw arguments string or raises."""
    response = client.chat.completions.create(
        model       = AZURE_DEPLOYMENT_NAME,
        messages    = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": f"Extract from:\n\n{document_text}"},
        ],
        tools        = [EXTRACTION_FUNCTION_SCHEMA],
        tool_choice  = {"type": "function", "function": {"name": "extract_tax_document"}},
        temperature  = 0.0,
        max_tokens   = 800,
    )
    tool_calls = response.choices[0].message.tool_calls
    if not tool_calls:
        raise ValueError("No function call in response")
    return tool_calls[0].function.arguments


def run_pipeline(document_text, doc_id=0, verbose=True):
    """
    Full extraction pipeline with retry, validation, and routing.
    Returns a pipeline_record dict.
    """
    record = {
        "doc_id":       doc_id,
        "timestamp":    datetime.datetime.now().isoformat(),
        "route":        RouteOutcome.FAILED,
        "extraction":   None,
        "error":        None,
        "retry_used":   False,
    }

    # ── Stage 1: First extraction attempt ─────────────────────────────────
    try:
        raw = _call_extraction_api(document_text, EXTRACTION_SYSTEM_PROMPT)
        data = json.loads(raw)
    except (json.JSONDecodeError, ValueError) as e:
        if verbose: print(f"  ⚠️  First attempt parse error: {e} — retrying...")

        # ── Stage 2: Retry with stricter prompt ───────────────────────────
        try:
            raw = _call_extraction_api(document_text, STRICT_EXTRACTION_PROMPT)
            data = json.loads(raw)
            record["retry_used"] = True
        except Exception as e2:
            record["error"] = f"Parse failed after retry: {e2}"
            record["route"] = RouteOutcome.FAILED
            if verbose: print(f"  ❌  Failed after retry. Logged.")
            return record

    # ── Stage 3: Pydantic validation ──────────────────────────────────────
    try:
        result = TaxDocument.model_validate(data)
    except ValidationError as e:
        # Partial recovery: mark bad fields as missing
        bad_fields = [str(err["loc"][0]) for err in e.errors() if err["loc"]]
        for bf in bad_fields:
            if bf in data:
                data[bf] = None
                if bf not in data.get("missing_fields", []):
                    data.setdefault("missing_fields", []).append(bf)
        # Lower confidence on validation failure
        data["extraction_confidence"] = min(data.get("extraction_confidence", 0.5), 0.60)
        try:
            result = TaxDocument.model_validate(data)
            if verbose: print(f"  ⚠️  Partial recovery after validation error on fields: {bad_fields}")
        except Exception as e3:
            record["error"] = f"Validation failed: {e3}"
            record["route"] = RouteOutcome.FAILED
            return record

    # ── Stage 4: Route ────────────────────────────────────────────────────
    record["extraction"] = result
    if result.human_review_required:
        record["route"] = RouteOutcome.HUMAN_REVIEW
    else:
        record["route"] = RouteOutcome.DB_INSERT

    return record


print("✅ run_pipeline() ready — retry + validate + route")

---
## 🧪 Step 5: Define the 5 Test Documents (same as 4B)

In [ ]:
TEST_DOCUMENTS = [
    {"id": 1, "label": "Clean email — well-structured",
     "text": """
From: priya.nair@techsolutions.in  Subject: GST filing Q3 FY2025
TechSolutions Pvt Ltd has completed services worth Rs 8,50,000 for a registered client
in Bengaluru during Q3 FY2025. Both parties are GST-registered in Karnataka.
This is a B2B intra-state transaction. Please confirm GST rate and filing timeline."""},

    {"id": 2, "label": "Ambiguous letter — approximate figures",
     "text": """
From: rajesh.kumar@alphacorp.in  Subject: GST query - urgent
Alpha Corp Pvt Ltd has done consulting work for a Dubai client — roughly 12 lakhs or so
(haven't finalised the invoice). Also local work in Karnataka, maybe 8 lakhs?
Tax period should be Q3 FY25 but could be Q2 also. Please advise."""},

    {"id": 3, "label": "Invoice snippet — missing period",
     "text": """
INVOICE — GlobalServ India Pvt Ltd, Mumbai → MNC Holdings LLC, New York, USA
Service: IT infrastructure consulting  Amount: USD 15,000
IGST: 0% (LUT filed - Ref: LUT/2024-25/0042)  Payment: 30 days"""},

    {"id": 4, "label": "Multi-entity memo — two transactions",
     "text": """
Internal Tax Memo October 2024:
1. Infovista Systems Ltd (Hyderabad) raised Rs 12,00,000 invoice to DataCore Pvt Ltd (Chennai)
   for software implementation. Inter-state B2B. Period: Q2 FY2025.
2. Infovista received legal advisory from advocate K.P. Sharma (individual) for Rs 75,000.
   RCM applies. Both need reflection in October GSTR-3B."""},

    {"id": 5, "label": "Near-empty message",
     "text": """Hi, can you check if there are any GST issues with the recent payments? - Amit"""},
]

print(f"✅ {len(TEST_DOCUMENTS)} test documents ready")

---
## 🚀 Step 6: Run Full Pipeline on All Documents

In [ ]:
print("RUNNING FULL PIPELINE")
print("=" * 65)

pipeline_records = []

for doc in TEST_DOCUMENTS:
    print(f"\nDoc {doc['id']}: {doc['label']}")
    record = run_pipeline(doc["text"], doc_id=doc["id"], verbose=True)
    pipeline_records.append(record)

    r = record["extraction"]
    route = record["route"]
    retry = " (retry used)" if record["retry_used"] else ""

    if r:
        conf_bar = "█" * int(r.extraction_confidence * 10)
        print(f"  Route:      {route.value.upper()}{retry}")
        print(f"  Confidence: {conf_bar} {r.extraction_confidence:.2f}")
        if r.missing_fields:
            print(f"  Missing:    {r.missing_fields}")
    else:
        print(f"  Route: FAILED — {record['error']}")


db_queue     = [rec for rec in pipeline_records if rec["route"] == RouteOutcome.DB_INSERT]
review_queue = [rec for rec in pipeline_records if rec["route"] == RouteOutcome.HUMAN_REVIEW]
failed       = [rec for rec in pipeline_records if rec["route"] == RouteOutcome.FAILED]

print("\n" + "=" * 65)
print(f"DB insert queue    : {len(db_queue)} documents")
print(f"Human review queue : {len(review_queue)} documents")
print(f"Failed             : {len(failed)} documents")

---
## 🗄️ Step 7: Mock DB Insert — High-Confidence Records

In [ ]:
compliance_db = []   # In production: INSERT INTO compliance_records

print("MOCK DB INSERT — High-Confidence Records")
print("-" * 55)

for rec in db_queue:
    r = rec["extraction"]
    db_record = {
        "record_id":         f"CR-{rec['doc_id']:04d}-{datetime.date.today().strftime('%Y%m%d')}",
        "entity_name":       r.entity_name,
        "tax_period":        r.tax_period,
        "transaction_type":  r.transaction_type,
        "jurisdiction":      r.jurisdiction,
        "liability_amount":  r.liability_amount,
        "currency":          r.currency,
        "applicable_section":r.applicable_section,
        "confidence":        r.extraction_confidence,
        "inserted_at":       datetime.datetime.now().isoformat(),
        "source_doc_id":     rec["doc_id"],
    }
    compliance_db.append(db_record)
    print(f"  ✅  Inserted: {db_record['record_id']} — {r.entity_name} / {r.tax_period}")

print(f"\nTotal records in mock DB: {len(compliance_db)}")

---
## 📬 Step 8: Auto-Generate Follow-Up Emails for Missing Fields

In [ ]:
FOLLOWUP_FIELD_QUESTIONS = {
    "tax_period":       "Could you confirm the exact tax period (e.g. Q3 FY2025 or October 2024)?",
    "liability_amount": "Could you provide the exact invoice or transaction amount (not approximate)?",
    "entity_name":      "Could you confirm the full legal entity name for the transaction?",
    "transaction_type": "Could you clarify the nature of the transaction (domestic B2B, export of services, etc.)?",
    "jurisdiction":     "Could you specify the state(s) or country/countries involved?",
    "currency":         "Could you confirm the currency of the transaction?",
}


def generate_followup_email(record):
    """Generate a targeted follow-up email asking only for missing fields."""
    r = record["extraction"]
    if not r or not r.missing_fields:
        return None

    questions = [
        f"  {i+1}. {FOLLOWUP_FIELD_QUESTIONS.get(field, f'Could you provide details on: {field}')}"
        for i, field in enumerate(r.missing_fields)
        if field in FOLLOWUP_FIELD_QUESTIONS
    ]

    if not questions:
        return None

    entity = r.entity_name or "your entity"
    email = (
        f"Subject: Follow-up — Additional details needed for GST filing ({entity})\n\n"
        f"Dear Client,\n\n"
        f"Thank you for your query. We have reviewed your message and require the following "
        f"additional details to proceed with your GST compliance:\n\n"
        + "\n".join(questions)
        + "\n\nPlease reply at your earliest convenience so we can complete the filing without delay.\n\n"
        + "Regards,\nTax Compliance Team"
    )
    return email


print("AUTO-GENERATED FOLLOW-UP EMAILS")
print("=" * 65)
for rec in review_queue:
    email = generate_followup_email(rec)
    if email:
        print(f"\n--- Doc {rec['doc_id']}: {TEST_DOCUMENTS[rec['doc_id']-1]['label']} ---")
        print(email)

---
## 💾 Step 9: Export Full Audit Trail

In [ ]:
audit_trail = {
    "pipeline":       "Day4_Extraction_Pipeline",
    "exported_at":    datetime.datetime.now().isoformat(),
    "model":          AZURE_DEPLOYMENT_NAME,
    "summary": {
        "total_documents":   len(TEST_DOCUMENTS),
        "db_inserted":       len(db_queue),
        "human_review":      len(review_queue),
        "failed":            len(failed),
    },
    "db_records":    compliance_db,
    "review_queue":  [
        {
            "doc_id":    rec["doc_id"],
            "label":     TEST_DOCUMENTS[rec["doc_id"]-1]["label"],
            "extraction":rec["extraction"].model_dump() if rec["extraction"] else None,
            "followup_email": generate_followup_email(rec),
        }
        for rec in review_queue
    ],
    "failed_docs":   [{"doc_id": r["doc_id"], "error": r["error"]} for r in failed],
    "raw_pipeline_records": [
        {
            "doc_id":      rec["doc_id"],
            "route":       rec["route"].value,
            "retry_used":  rec["retry_used"],
            "error":       rec["error"],
            "extraction":  rec["extraction"].model_dump() if rec["extraction"] else None,
        }
        for rec in pipeline_records
    ]
}

with open("day4_extractions.json", "w", encoding="utf-8") as f:
    json.dump(audit_trail, f, indent=2, ensure_ascii=False, default=str)

print("✅ Full audit trail saved to day4_extractions.json")
print(f"\nSummary:")
for k, v in audit_trail["summary"].items():
    print(f"  {k:<22}: {v}")

---
## 🎓 The Full Day 4 Arc

```
Day 2:  ReAct agent  →  implicit reasoning, tool calls
Day 3:  CoT + few-shot  →  explicit reasoning, structured JSON from clean questions
Day 4:  Function calling  →  structured extraction from unstructured documents

The Pydantic pattern grew:
  TaxClassification (Day 3)   — clean input, all fields present
  TaxDocument (Day 4)         — messy input, Optional fields, missing tracker, graceful degradation
```

**What you've built across the three Day 4 notebooks:**

| Notebook | Capability |
|---|---|
| 4A | Robust schema with Optional fields, validators, confidence design |
| 4B | Function calling extractor on 5 real-world documents |
| 4C | Full pipeline: retry · validate · route · DB insert · follow-up emails · audit trail |

---
## ➡️ Day 5: RAG Foundations & Vector Stores

**The last gap:** so far you have been handing the agent the document directly.  
In Day 5, we build a vector store from tax circulars and guidance documents  
so the agent can **retrieve the right document automatically** before extracting or answering.  
That's Retrieval-Augmented Generation — and it eliminates the need for you to know which document to pass in.